# 02 · AfriMed Assistant: train and evaluate

This notebook runs every experiment and builds the experiments table. Each experiment is evaluated on **val and test** with the same code:

| Run | Model | Training | Purpose |
|---|---|---|---|
| `E0_zeroshot_0.5B` | Qwen2.5-0.5B-Instruct | none | **baseline** |
| `E0_zeroshot_1.5B` | Qwen2.5-1.5B-Instruct | none | baseline for the size comparison |
| `E1_lora_0.5B_saq+mcq` | 0.5B | LoRA on SAQ + MCQ | **main model** |
| `E2_lora_0.5B_saq` | 0.5B | LoRA on SAQ only | training-data experiment |
| `E3_lora_1.5B_saq+mcq` | 1.5B | LoRA on SAQ + MCQ | model-size experiment |

**Settings** (from `docs/benchmark_t4.md`): max_len 512, micro-batch 4 × grad-accum 4 = effective batch 16, 3 epochs, LoRA r=16/alpha=32 on all projections, lr 2e-4 with cosine decay and 5% warm-up, fp16. The adapter with the best val loss is kept.

**Metrics**
- MCQ: exact-set accuracy. All correct options must be chosen and no wrong ones, which matters for multi-answer questions. The parse-failure rate is reported too.
- SAQ: ROUGE-L and BERTScore F1 (roberta-large, rescaled).
- All headline numbers come with 95% bootstrap confidence intervals and slices by tier, single/multi answer, and questions that need African expertise.

**How to run**
1. *Runtime → Change runtime type → T4 GPU*, then *Run all*.
2. Allow Google Drive access. Results go to `MyDrive/afrimed-qa-assistant/runs/`.
3. Upload `qa_train.jsonl`, `qa_val.jsonl` and `qa_test.jsonl` from `data/processed/` when asked. They are copied to Drive, so you only upload them once.
4. Finished experiments are **skipped** on rerun, so after a disconnect, just *Run all* again. An experiment that was interrupted restarts from the beginning.

The total run time is about 1.5–2 hours. Send back `runs/experiments_table.csv` and the `metrics.json` from each run folder; `predictions_*.jsonl` is for error analysis.

*Generated by `scripts/build_train_notebook.py` from `src/afrimed/`, so don't hand-edit this notebook.*

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q peft rouge-score bert-score
# Colab ships torchao 0.10; recent peft raises ImportError if an older torchao is present. We don't use it.
!pip uninstall -y -q torchao

## Storage (Google Drive)

In [ ]:
import os
USE_DRIVE = True   # False -> results in /content only (lost when the runtime ends)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/afrimed-qa-assistant"
else:
    BASE = "/content/afrimed-qa-assistant"
DATA_DIR, OUT_ROOT = f"{BASE}/data", f"{BASE}/runs"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_ROOT, exist_ok=True)
print(BASE)

## Project code (`src/afrimed/`)

In [ ]:
!mkdir -p /content/afrimed_src/afrimed

In [ ]:
%%writefile /content/afrimed_src/afrimed/__init__.py
"""AfriMed Assistant: shared code for training, evaluation and the Gradio app."""

In [ ]:
%%writefile /content/afrimed_src/afrimed/prompts.py
"""Prompt format shared by training, evaluation and the app.

MCQ: options shown as A-E; target is "Answer: B" (or "Answer: B, D" for multi-answer).
SAQ: target is the reference answer text.
"""
import re

SYSTEM = ("You are a medical question-answering assistant focused on healthcare in African contexts. "
          "Answer accurately and concisely. You do not replace a qualified health professional.")
LETTERS = "ABCDE"


def option_letter(key: str) -> str:
    return LETTERS[int(key.removeprefix("option")) - 1]


def mcq_user(rec: dict) -> str:
    opts = "\n".join(f"{option_letter(k)}. {v}" for k, v in sorted(rec["options"].items()))
    return (f"{rec['question']}\n\nOptions:\n{opts}\n\n"
            "Select the correct option(s). Reply in the form 'Answer: <letter(s)>'.")


def gold_letters(rec: dict) -> frozenset:
    return frozenset(option_letter(k) for k in rec["answer"])


def mcq_target(rec: dict) -> str:
    return "Answer: " + ", ".join(sorted(gold_letters(rec)))


def to_messages(rec: dict, with_answer: bool = True) -> list[dict]:
    """rec needs question_type and question (+ options for MCQ); answer only if with_answer."""
    user = mcq_user(rec) if rec["question_type"] == "mcq" else rec["question"]
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    if with_answer:
        target = mcq_target(rec) if rec["question_type"] == "mcq" else rec["answer"]
        msgs.append({"role": "assistant", "content": target})
    return msgs


# A leading run of letters such as "B", "(B)", "B, D", "B and D", "B/D".
_LETTER_RUN = re.compile(r"\(?([A-E])\)?((?:\s*(?:,\s*and|,|and|&|/|or)\s*\(?[A-E]\)?)*)(?![A-Za-z])")
_ANSWER_CUE = re.compile(r"(?i)answer\s*(?:is)?\s*[:\-]?\s*")


def _leading_letters(text: str, require_boundary: bool) -> frozenset | None:
    m = _LETTER_RUN.match(text)
    if not m:
        return None
    rest = text[m.end():]
    # Without an "Answer:" cue, "A 35-year-old..." must not parse as option A.
    if require_boundary and rest and not re.match(r"\s*(?:$|[.):\n])", rest):
        return None
    return frozenset(re.findall(r"[A-E]", m.group(0)))


def parse_mcq(text: str) -> frozenset | None:
    """Extract the chosen option letters from model output; None if unparseable."""
    text = text.replace("*", "").strip()
    for cue in _ANSWER_CUE.finditer(text):
        letters = _leading_letters(text[cue.end():], require_boundary=False)
        if letters:
            return letters
    letters = _leading_letters(text, require_boundary=True)
    if letters:
        return letters
    mentioned = set(re.findall(r"\b[Oo]ption\s+([A-E])\b", text))
    return frozenset(mentioned) if len(mentioned) == 1 else None

In [ ]:
%%writefile /content/afrimed_src/afrimed/data.py
"""Loading the processed splits and turning records into training tensors."""
import json

import torch

from .prompts import to_messages


def load_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def write_jsonl(rows: list[dict], path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def encode(tok, rec: dict, max_len: int) -> dict:
    """Token ids with labels masked (-100) on the prompt, so loss is on the answer only."""
    # Render to text first: apply_chat_template's tokenize=True return type differs across versions.
    prompt = tok.apply_chat_template(to_messages(rec, False), tokenize=False, add_generation_prompt=True)
    full = tok.apply_chat_template(to_messages(rec), tokenize=False)
    assert full.startswith(prompt), "chat template is not prefix-stable"
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids[:max_len], "labels": labels[:max_len], "truncated": len(f_ids) > max_len}


def collate(examples: list[dict], pad_id: int) -> dict:
    n = max(len(e["input_ids"]) for e in examples)
    ids = torch.full((len(examples), n), pad_id)
    labels = torch.full((len(examples), n), -100)
    mask = torch.zeros((len(examples), n), dtype=torch.long)
    for i, e in enumerate(examples):
        k = len(e["input_ids"])
        ids[i, :k] = torch.tensor(e["input_ids"])
        labels[i, :k] = torch.tensor(e["labels"])
        mask[i, :k] = 1
    return {"input_ids": ids, "labels": labels, "attention_mask": mask}

In [ ]:
%%writefile /content/afrimed_src/afrimed/infer.py
"""Batched greedy generation, shared by evaluation and the app."""
import torch

from .prompts import to_messages

# Qwen2.5's own generation_config uses repetition_penalty 1.05; it is kept, but made explicit.
GEN_KWARGS = {"do_sample": False, "repetition_penalty": 1.05, "use_cache": True}


def build_prompt(tok, rec: dict) -> str:
    return tok.apply_chat_template(to_messages(rec, with_answer=False), tokenize=False,
                                   add_generation_prompt=True)


@torch.no_grad()
def generate(model, tok, prompts: list[str], max_new_tokens: int, batch_size: int = 16) -> list[str]:
    """Greedy-decode each prompt; returns texts in input order."""
    model.eval()
    old_side, tok.padding_side = tok.padding_side, "left"
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))  # similar lengths per batch
    out = [None] * len(prompts)
    try:
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id,
                                 **GEN_KWARGS)
            for i, row in zip(idx, gen[:, enc["input_ids"].shape[1]:]):
                out[i] = tok.decode(row, skip_special_tokens=True).strip()
    finally:
        tok.padding_side = old_side
    return out

In [ ]:
%%writefile /content/afrimed_src/afrimed/metrics.py
"""Evaluation metrics: MCQ exact-set accuracy, ROUGE-L, BERTScore, bootstrap CIs."""
import numpy as np


def bootstrap_ci(values, n: int = 1000, seed: int = 0) -> list[float] | None:
    values = np.asarray([v for v in values if v is not None], dtype=float)
    if len(values) == 0:
        return None
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), (n, len(values)))].mean(axis=1)
    return [round(float(np.percentile(means, 2.5)), 4), round(float(np.percentile(means, 97.5)), 4)]


def mcnemar_exact(a: list[bool], b: list[bool]) -> dict:
    """Paired comparison of two models on the same items (exact two-sided McNemar / binomial test).

    a, b: per-item correctness, aligned by item. Only discordant pairs carry information.
    """
    from math import comb
    assert len(a) == len(b)
    only_a = sum(x and not y for x, y in zip(a, b))
    only_b = sum(y and not x for x, y in zip(a, b))
    n = only_a + only_b
    k = min(only_a, only_b)
    p = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0
    return {"n": len(a), "acc_a": round(sum(a) / len(a), 4), "acc_b": round(sum(b) / len(b), 4),
            "only_a_correct": only_a, "only_b_correct": only_b, "p_value": round(p, 5)}


def rouge_l(preds: list[str], refs: list[str]) -> list[float]:
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    return [scorer.score(r, p)["rougeL"].fmeasure for p, r in zip(preds, refs)]


def bert_score_f1(preds: list[str], refs: list[str]) -> tuple[list[float] | None, str | None]:
    """roberta-large BERTScore F1, rescaled with the package baseline. Returns (scores, error)."""
    try:
        from bert_score import score
        _, _, f1 = score([p or " " for p in preds], refs, lang="en", model_type="roberta-large",
                         rescale_with_baseline=True, batch_size=32, verbose=False)
        return f1.tolist(), None
    except Exception as e:  # keep the other metrics if BERTScore breaks on a library update
        return None, f"{type(e).__name__}: {e}"


def _mean(xs):
    xs = [x for x in xs if x is not None]
    return round(float(np.mean(xs)), 4) if xs else None


def summarise(rows: list[dict]) -> dict:
    """rows: per-item predictions with question_type, tier, multi_answer, scores. Returns metrics + slices."""
    mcq = [r for r in rows if r["question_type"] == "mcq"]
    saq = [r for r in rows if r["question_type"] == "saq"]
    out = {"n_mcq": len(mcq), "n_saq": len(saq)}

    def mcq_block(items):
        return {"n": len(items), "acc": _mean([r["correct"] for r in items]),
                "parse_fail": _mean([r["parsed"] is None for r in items])}

    out["mcq"] = {**mcq_block(mcq), "acc_ci95": bootstrap_ci([r["correct"] for r in mcq])}
    out["mcq_by"] = {
        "single": mcq_block([r for r in mcq if not r["multi_answer"]]),
        "multi": mcq_block([r for r in mcq if r["multi_answer"]]),
        **{f"tier={t}": mcq_block([r for r in mcq if r["tier"] == t]) for t in ("expert", "crowdsourced")},
        "african_expertise": mcq_block([r for r in mcq if r["requires_african_expertise"]]),
    }

    def saq_block(items):
        return {"n": len(items),
                "rougeL": _mean([r["rougeL"] for r in items]),
                "bertscore_f1": _mean([r.get("bertscore_f1") for r in items]),
                "pred_words": _mean([len(r["prediction"].split()) for r in items]),
                "ref_words": _mean([len(r["reference"].split()) for r in items])}

    out["saq"] = {**saq_block(saq),
                  "rougeL_ci95": bootstrap_ci([r["rougeL"] for r in saq]),
                  "bertscore_ci95": bootstrap_ci([r.get("bertscore_f1") for r in saq])}
    out["saq_by"] = {
        **{f"tier={t}": saq_block([r for r in saq if r["tier"] == t]) for t in ("expert", "crowdsourced")},
        "african_expertise": saq_block([r for r in saq if r["requires_african_expertise"]]),
    }
    return out

In [ ]:
%%writefile /content/afrimed_src/afrimed/train.py
"""LoRA fine-tuning loop (fp16 autocast, for a T4). Settings come from docs/benchmark_t4.md."""
import math
import random
import time
from dataclasses import asdict, dataclass, field

import torch
from peft import LoraConfig, get_peft_model, load_peft_weights, set_peft_model_state_dict
from transformers import AutoModelForCausalLM

from .data import collate, encode


@dataclass
class TrainConfig:
    model_name: str
    data_mix: str = "saq+mcq"          # "saq+mcq" or "saq"
    max_len: int = 512
    micro_batch: int = 4
    grad_accum: int = 4                # effective batch 16
    epochs: int = 3
    lr: float = 2e-4
    warmup_ratio: float = 0.05
    max_grad_norm: float = 1.0
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    seed: int = 42

    def to_dict(self):
        return asdict(self)


def select_mix(records: list[dict], data_mix: str) -> list[dict]:
    types = {"saq+mcq": {"saq", "mcq"}, "saq": {"saq"}}[data_mix]
    return [r for r in records if r["question_type"] in types]


def load_base(name: str, dtype=torch.float16, dtype_kw: str = "dtype"):
    return AutoModelForCausalLM.from_pretrained(name, **{dtype_kw: dtype}).cuda()


def add_lora(model, cfg: TrainConfig):
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=cfg.lora_dropout, bias="none",
        task_type="CAUSAL_LM", target_modules=cfg.target_modules))
    for p in model.parameters():          # fp32 adapters for stable fp16 mixed precision
        if p.requires_grad:
            p.data = p.data.float()
    return model


@torch.no_grad()
def eval_loss(model, encoded: list[dict], pad_id: int, batch_size: int) -> float:
    """Token-weighted mean loss over answer tokens."""
    model.eval()
    total, n_tok = 0.0, 0
    for i in range(0, len(encoded), batch_size):
        batch = {k: v.cuda() for k, v in collate(encoded[i:i + batch_size], pad_id).items()}
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**batch).loss
        n = int((batch["labels"][:, 1:] != -100).sum())
        total += loss.item() * n
        n_tok += n
    model.train()
    return total / max(n_tok, 1)


def train(model, tok, train_recs: list[dict], val_recs: list[dict], cfg: TrainConfig, adapter_dir: str,
          log=print) -> dict:
    """Train; saves the adapter with the best val loss to adapter_dir and restores it at the end."""
    rng = random.Random(cfg.seed)
    torch.manual_seed(cfg.seed)
    train_enc = [encode(tok, r, cfg.max_len) for r in select_mix(train_recs, cfg.data_mix)]
    val_enc = [encode(tok, r, cfg.max_len) for r in select_mix(val_recs, cfg.data_mix)]
    log(f"train examples {len(train_enc)} (truncated {sum(e['truncated'] for e in train_enc)}), "
        f"val examples {len(val_enc)}")

    steps_per_epoch = math.ceil(len(train_enc) / cfg.micro_batch / cfg.grad_accum)
    total_steps = steps_per_epoch * cfg.epochs
    warmup = max(1, int(cfg.warmup_ratio * total_steps))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg.lr)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warmup if s < warmup else
                                              0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total_steps - warmup))))
    scaler = torch.amp.GradScaler("cuda")

    history = {"steps_per_epoch": steps_per_epoch, "total_steps": total_steps, "train_loss": [], "val": []}
    best, step, t0 = float("inf"), 0, time.time()
    model.train()
    for epoch in range(1, cfg.epochs + 1):
        order = list(range(len(train_enc)))
        rng.shuffle(order)
        batches = [order[i:i + cfg.micro_batch] for i in range(0, len(order), cfg.micro_batch)]
        running = []
        for b_idx, b in enumerate(batches):
            batch = {k: v.cuda() for k, v in collate([train_enc[i] for i in b], tok.pad_token_id).items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss
            scaler.scale(loss / cfg.grad_accum).backward()
            running.append(loss.item())
            if (b_idx + 1) % cfg.grad_accum == 0 or b_idx == len(batches) - 1:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, cfg.max_grad_norm)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                step += 1
                if step % 25 == 0:
                    avg = sum(running) / len(running)
                    history["train_loss"].append({"step": step, "loss": round(avg, 4)})
                    log(f"epoch {epoch} step {step}/{total_steps} loss {avg:.4f} "
                        f"lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s")
                    running = []
        v = eval_loss(model, val_enc, tok.pad_token_id, cfg.micro_batch * 2)
        improved = v < best
        history["val"].append({"epoch": epoch, "step": step, "val_loss": round(v, 4), "best": improved})
        log(f"== epoch {epoch} val_loss {v:.4f}{' (best, saved)' if improved else ''}")
        if improved:
            best = v
            model.save_pretrained(adapter_dir)
            tok.save_pretrained(adapter_dir)

    best_epoch = min(history["val"], key=lambda h: h["val_loss"])["epoch"]
    if best_epoch != cfg.epochs:
        set_peft_model_state_dict(model, load_peft_weights(adapter_dir))
        log(f"restored best adapter from epoch {best_epoch}")
    history.update(best_epoch=best_epoch, best_val_loss=round(best, 4), train_minutes=round((time.time() - t0) / 60, 1))
    model.eval()
    return history

In [ ]:
%%writefile /content/afrimed_src/afrimed/evaluate.py
"""Run a model over a split and score it (MCQ exact-set accuracy; SAQ ROUGE-L + BERTScore)."""
import gc

import torch

from .infer import build_prompt, generate
from .metrics import bert_score_f1, bootstrap_ci, rouge_l, summarise
from .prompts import LETTERS, gold_letters, parse_mcq

MCQ_MAX_NEW = 48     # room for a zero-shot model that explains before giving the letter
SAQ_MAX_NEW = 256    # SAQ references: median ~29 words, a few expert essays far longer


def predict(model, tok, records: list[dict], batch_size: int = 16, log=print) -> list[dict]:
    rows = []
    for qtype, max_new in (("mcq", MCQ_MAX_NEW), ("saq", SAQ_MAX_NEW)):
        recs = [r for r in records if r["question_type"] == qtype]
        log(f"generating {len(recs)} {qtype} answers ...")
        bs = batch_size * 2 if qtype == "mcq" else batch_size
        preds = generate(model, tok, [build_prompt(tok, r) for r in recs], max_new, bs)
        for r, p in zip(recs, preds):
            row = {k: r[k] for k in ("id", "question_type", "tier", "specialty", "country",
                                     "requires_african_expertise", "multi_answer", "question")}
            row["prediction"] = p
            if qtype == "mcq":
                parsed = parse_mcq(p)
                row.update(options=r["options"], gold=sorted(gold_letters(r)),
                           parsed=sorted(parsed) if parsed else None,
                           correct=parsed == gold_letters(r))
            else:
                row["reference"] = r["answer"]
            rows.append(row)
    return rows


@torch.no_grad()
def mcq_option_likelihood(model, tok, records: list[dict], batch_size: int = 16) -> tuple[list[dict], dict]:
    """Format-independent MCQ score: the option letter the model rates most likely right after "Answer:".

    Generative exact-set scoring also measures whether a model follows the answer format (zero-shot models
    often list several letters); this isolates which option the model prefers. It picks one letter, so the
    headline is accuracy on single-answer questions.
    """
    letter_ids = [sorted({tok(f" {c}", add_special_tokens=False)["input_ids"][0],
                          tok(c, add_special_tokens=False)["input_ids"][0]}) for c in LETTERS]
    recs = [r for r in records if r["question_type"] == "mcq"]
    prompts = [build_prompt(tok, r) + "Answer:" for r in recs]
    model.eval()
    old_side, tok.padding_side = tok.padding_side, "left"
    rows = []
    try:
        for i in range(0, len(recs), batch_size):
            enc = tok(prompts[i:i + batch_size], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(model.device)
            # With left padding a plain forward needs explicit positions (generate() does this itself).
            pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**enc, position_ids=pos, logits_to_keep=1).logits[:, -1].float()
            for r, lg in zip(recs[i:i + batch_size], logits):
                scores = [max(lg[t].item() for t in ids) for ids in letter_ids]
                choice = LETTERS[max(range(len(LETTERS)), key=scores.__getitem__)]
                gold = gold_letters(r)
                rows.append({"id": r["id"], "tier": r["tier"], "multi_answer": r["multi_answer"],
                             "requires_african_expertise": r["requires_african_expertise"],
                             "choice": choice, "gold": sorted(gold), "choice_in_gold": choice in gold})
    finally:
        tok.padding_side = old_side

    def acc(items):
        return {"n": len(items),
                "acc": round(sum(r["choice_in_gold"] for r in items) / len(items), 4) if items else None}

    single = [r for r in rows if not r["multi_answer"]]
    summary = {
        "single": {**acc(single), "acc_ci95": bootstrap_ci([r["choice_in_gold"] for r in single])},
        "single_by": {
            **{f"tier={t}": acc([r for r in single if r["tier"] == t]) for t in ("expert", "crowdsourced")},
            "african_expertise": acc([r for r in single if r["requires_african_expertise"]]),
        },
        # For multi-answer questions: is the top choice at least one of the correct options?
        "multi_top_choice_in_gold": acc([r for r in rows if r["multi_answer"]]),
    }
    return rows, summary


def score(rows: list[dict], log=print) -> dict:
    """Adds ROUGE-L/BERTScore to SAQ rows in place and returns the summary. Free the LLM before calling."""
    saq = [r for r in rows if r["question_type"] == "saq"]
    for r, s in zip(saq, rouge_l([r["prediction"] for r in saq], [r["reference"] for r in saq])):
        r["rougeL"] = round(s, 4)
    gc.collect()
    torch.cuda.empty_cache()
    log("computing BERTScore ...")
    f1, err = bert_score_f1([r["prediction"] for r in saq], [r["reference"] for r in saq])
    if f1 is not None:
        for r, s in zip(saq, f1):
            r["bertscore_f1"] = round(s, 4)
    summary = summarise(rows)
    summary["bertscore_error"] = err
    return summary

In [ ]:
%%writefile /content/afrimed_src/afrimed/experiments.py
"""The experiment grid. Run folders under runs/ are named by these keys."""
M05, M15 = "Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct"

EXPERIMENTS = {
    "E0_zeroshot_0.5B":     {"model": M05, "lora": False},
    "E0_zeroshot_1.5B":     {"model": M15, "lora": False},
    "E1_lora_0.5B_saq+mcq": {"model": M05, "lora": True, "data_mix": "saq+mcq"},
    "E2_lora_0.5B_saq":     {"model": M05, "lora": True, "data_mix": "saq"},
    "E3_lora_1.5B_saq+mcq": {"model": M15, "lora": True, "data_mix": "saq+mcq"},
}

In [ ]:
%%writefile /content/afrimed_src/afrimed/gate.py
"""Medical vs non-medical question gate: TF-IDF features + logistic regression.

Scope policy (docs/ood_gate.md): in-domain = questions seeking information about human health, disease,
symptoms, treatment, medicines, prevention, nutrition for health, mental, reproductive and maternal health,
and public health. Anything else (including veterinary, health-sector jobs, prices and admin) is declined.
Only scikit-learn is needed, so the app can import this without torch.
"""
import joblib
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline

FEATURE_SETS = ("word", "char", "word+char")


def build_pipeline(features: str = "word+char", C: float = 1.0, seed: int = 42) -> Pipeline:
    parts = []
    if "word" in features:
        parts.append(("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)))
    if "char" in features:
        parts.append(("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                                              sublinear_tf=True, max_features=200_000)))
    return Pipeline([
        ("features", FeatureUnion(parts)),
        ("clf", LogisticRegression(C=C, max_iter=3000, class_weight="balanced", random_state=seed)),
    ])


class MedicalGate:
    """Wraps a fitted pipeline and the decision threshold chosen on validation data."""

    def __init__(self, pipeline: Pipeline, threshold: float, meta: dict | None = None):
        self.pipeline, self.threshold, self.meta = pipeline, threshold, meta or {}

    def prob_medical(self, texts: list[str]) -> list[float]:
        return self.pipeline.predict_proba(texts)[:, 1].tolist()

    def is_medical(self, text: str) -> tuple[bool, float]:
        p = self.prob_medical([text])[0]
        return p >= self.threshold, p

    def save(self, path) -> None:
        joblib.dump({"pipeline": self.pipeline, "threshold": self.threshold,
                     "meta": {**self.meta, "sklearn_version": sklearn.__version__}}, path)

    @classmethod
    def load(cls, path) -> "MedicalGate":
        d = joblib.load(path)
        return cls(d["pipeline"], d["threshold"], d["meta"])

In [ ]:
%%writefile /content/afrimed_src/afrimed/assistant.py
"""The assistant pipeline used by the app: crisis check -> out-of-domain gate -> fine-tuned model.

The crisis check is a rule, not a model, and runs first: the gate was measured to decline some suicide-related
questions (docs/ood_gate.md), so a crisis message must never depend on it.
"""
import re
import time
from pathlib import Path

from .infer import GEN_KWARGS, build_prompt

CRISIS = re.compile(
    r"\b(suicid\w*|kill(?:ing)? (?:my|him|her|them)sel(?:f|ves)|end(?:ing)? (?:my|his|her|their) (?:own )?life|"
    r"wants? to die|don'?t want to live|no reason to live|self[- ]?harm\w*|hurt(?:ing)? (?:my|him|her)self|"
    r"cutting (?:my|him|her)self|overdos\w*)\b", re.IGNORECASE)

CRISIS_MSG = (
    "**If you or someone you know may be thinking about suicide or self-harm, please reach out for help now.** "
    "Call your local emergency number or go to the nearest health facility. Free, confidential crisis lines "
    "for your country are listed at [findahelpline.com](https://findahelpline.com). If someone is in immediate "
    "danger, stay with them and get help.")

DECLINE_MSG = (
    "I can only answer questions about **human health and medicine**, such as symptoms, diseases, treatments, "
    "medicines, pregnancy, mental health or nutrition. This question looks outside that scope, so I won't answer "
    "it. If it is a health question, try rephrasing it with more detail about the health concern.")

DISCLAIMER = (
    "_This answer comes from a small AI model fine-tuned on AfriMed-QA. It can be wrong and is not medical "
    "advice. For diagnosis or treatment, see a qualified health professional. In an emergency, go to the "
    "nearest health facility._")


class Assistant:
    def __init__(self, model, tok, gate=None, max_new_tokens: int = 128):
        self.model, self.tok, self.gate, self.max_new_tokens = model, tok, gate, max_new_tokens

    @classmethod
    def load(cls, base_model: str, adapter: str | None = None, gate_path=None, threads: int | None = None,
             max_new_tokens: int = 128) -> "Assistant":
        """CPU load: fp32 base model, LoRA adapter merged in for faster generation."""
        import torch
        import transformers
        from packaging import version
        from transformers import AutoModelForCausalLM, AutoTokenizer

        from .gate import MedicalGate

        if threads:
            torch.set_num_threads(threads)
        dtype_kw = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
        # Training didn't change the tokenizer, so the base model's is identical if the adapter has none.
        has_tok = adapter and any((Path(adapter) / f).exists() for f in ("tokenizer.json", "tokenizer_config.json"))
        tok = AutoTokenizer.from_pretrained(adapter if has_tok else base_model)
        model = AutoModelForCausalLM.from_pretrained(base_model, **{dtype_kw: torch.float32})
        if adapter:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, adapter).merge_and_unload()
        model.eval()
        gate = MedicalGate.load(gate_path) if gate_path else None
        return cls(model, tok, gate, max_new_tokens)

    def _stream_answer(self, question: str):
        """Yield the answer text as it grows (greedy decoding, same settings as evaluation)."""
        from threading import Thread

        from transformers import TextIteratorStreamer

        prompt = build_prompt(self.tok, {"question_type": "saq", "question": question})
        enc = self.tok(prompt, return_tensors="pt", add_special_tokens=False)
        streamer = TextIteratorStreamer(self.tok, skip_prompt=True, skip_special_tokens=True)
        worker = Thread(target=self.model.generate, kwargs=dict(
            **enc, streamer=streamer, max_new_tokens=self.max_new_tokens, pad_token_id=self.tok.pad_token_id,
            **GEN_KWARGS))
        worker.start()
        text = ""
        for piece in streamer:
            text += piece
            yield text
        worker.join()

    def stream(self, question: str):
        """Yield partial results while the answer is generated; the last one is final.

        Each result: markdown to show, route (answered / declined / crisis+...), gate probability,
        elapsed seconds, the bare answer text, and done.
        """
        t0 = time.perf_counter()
        question = (question or "").strip()
        result = {"markdown": "Please type a health question.", "route": "empty", "prob_medical": None,
                  "seconds": 0.0, "answer": None, "done": True}
        if not question:
            yield result
            return
        is_med, p = self.gate.is_medical(question) if self.gate else (True, None)
        crisis = bool(CRISIS.search(question))
        head = [CRISIS_MSG] if crisis else []
        result.update(route=("crisis+" if crisis else "") + ("answered" if is_med else "declined"),
                      prob_medical=p, done=False)
        if is_med:
            for text in self._stream_answer(question):
                result.update(markdown="\n\n".join(head + [text]), answer=text,
                              seconds=round(time.perf_counter() - t0, 2))
                yield result
            tail = [result["answer"] or "", DISCLAIMER]
        else:
            tail = [] if crisis else [DECLINE_MSG]
        result.update(markdown="\n\n".join(head + tail), seconds=round(time.perf_counter() - t0, 2), done=True)
        yield result

    def respond(self, question: str) -> dict:
        """Non-streaming: the final result of stream()."""
        for result in self.stream(question):
            pass
        return result

## Imports, environment, self-checks

In [ ]:
import sys, gc, json, hashlib, platform, random, time, importlib, importlib.util
sys.path.insert(0, "/content/afrimed_src")
import numpy as np, pandas as pd, torch, transformers, peft
from packaging import version
from transformers import AutoTokenizer
import afrimed.prompts, afrimed.data, afrimed.infer, afrimed.metrics, afrimed.train, afrimed.evaluate, afrimed.experiments
for m in (afrimed.prompts, afrimed.data, afrimed.infer, afrimed.metrics, afrimed.train, afrimed.evaluate,
          afrimed.experiments):
    importlib.reload(m)   # pick up re-written modules without restarting
from afrimed.prompts import parse_mcq
from afrimed.data import load_jsonl, write_jsonl
from afrimed.train import TrainConfig, load_base, add_lora, train
from afrimed.evaluate import predict, score, mcq_option_likelihood
from afrimed.experiments import EXPERIMENTS

assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > T4 GPU"
assert importlib.util.find_spec("torchao") is None, "torchao still installed: Runtime > Restart session, then Run all"
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56") else "torch_dtype"
ENV = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
       "transformers": transformers.__version__, "peft": peft.__version__, "python": platform.python_version()}
print(ENV)

# Answer-parser self-checks (cheap; catches regressions before an hour of GPU time).
F = frozenset
for text, want in [("Answer: B", F("B")), ("Answer: B, D", F("BD")), ("**Answer:** C", F("C")),
                   ("The answer is (D).", F("D")), ("Answer: D. Simple goiter", F("D")),
                   ("Answer: A, C, and E", F("ACE")), ("B. Graves' disease", F("B")),
                   ("A 35-year-old woman ...", None), ("Vitamin A deficiency", None),
                   ("Answer: Both A and B", None), ("I think option C is right", F("C"))]:
    assert parse_mcq(text) == want, (text, parse_mcq(text), want)
print("parser self-checks passed")

## Data (our split from `scripts/prepare_data.py`, seed 42)

In [ ]:
EXPECTED_SHA256 = {
    "qa_train.jsonl": "17c8393f77d4d5773869a9063638532e843a08a932017725618ac342962f828e",
    "qa_val.jsonl": "fdefd6ac7619b53b689096bd5d0ffa63ae62596223450cece4ed7992a07aa0a9",
    "qa_test.jsonl": "c3066fc32696745d7a16e597b4ba8d7686031121998c45087cc059aaed974b6f",
}
missing = [f for f in EXPECTED_SHA256 if not os.path.exists(f"{DATA_DIR}/{f}")]
if missing:
    from google.colab import files
    print("Upload:", missing)
    for name, content in files.upload().items():
        with open(f"{DATA_DIR}/{os.path.basename(name)}", "wb") as fh:
            fh.write(content)
for f, expected in EXPECTED_SHA256.items():
    digest = hashlib.sha256(open(f"{DATA_DIR}/{f}", "rb").read()).hexdigest()
    assert digest == expected, f"{f}: hash mismatch - not the prepare_data.py seed-42 split"
TRAIN, VAL, TEST = (load_jsonl(f"{DATA_DIR}/qa_{s}.jsonl") for s in ("train", "val", "test"))
print(len(TRAIN), "train /", len(VAL), "val /", len(TEST), "test - hashes OK")

## Experiments

In [ ]:
print(json.dumps(EXPERIMENTS, indent=1))   # defined in src/afrimed/experiments.py
RUN = list(EXPERIMENTS)        # e.g. ["E0_zeroshot_0.5B", "E1_lora_0.5B_saq+mcq"] to run a subset
EPOCHS = 3
EVAL_SPLITS = {"val": VAL, "test": TEST}
SEED = 42

In [ ]:
def run_experiment(name, spec):
    out = f"{OUT_ROOT}/{name}"
    if os.path.exists(f"{out}/metrics.json"):
        print(f"[skip] {name} already finished"); return
    os.makedirs(out, exist_ok=True)
    logf = open(f"{out}/log.txt", "a", encoding="utf-8")
    def log(msg):
        line = f"[{time.strftime('%H:%M:%S')}] {msg}"
        print(line, flush=True); logf.write(line + "\n"); logf.flush()

    log(f"=== {name}: {spec}")
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(spec["model"])
    model = load_base(spec["model"], dtype_kw=DTYPE_KW)
    cfg, history = None, None
    if spec["lora"]:
        cfg = TrainConfig(model_name=spec["model"], data_mix=spec["data_mix"], epochs=EPOCHS, seed=SEED)
        model = add_lora(model, cfg)
        history = train(model, tok, TRAIN, VAL, cfg, f"{out}/adapter", log)

    preds, timing = {}, {}
    for split, recs in EVAL_SPLITS.items():
        t = time.time()
        preds[split] = predict(model, tok, recs, log=log)
        timing[f"{split}_generate_min"] = round((time.time() - t) / 60, 1)
    del model; gc.collect(); torch.cuda.empty_cache()

    metrics = {"experiment": name, "spec": spec, "train_config": cfg.to_dict() if cfg else None,
               "history": history, "timing": timing, "env": ENV}
    for split in EVAL_SPLITS:
        metrics[split] = score(preds[split], log)
        write_jsonl(preds[split], f"{out}/predictions_{split}.jsonl")
    with open(f"{out}/metrics.json", "w") as f:   # written last: marks the run as finished
        json.dump(metrics, f, indent=2)
    t = metrics["test"]
    log(f"TEST  MCQ acc {t['mcq']['acc']} {t['mcq']['acc_ci95']} (parse fail {t['mcq']['parse_fail']})  "
        f"SAQ ROUGE-L {t['saq']['rougeL']}  BERTScore {t['saq']['bertscore_f1']}")
    logf.close()

for name in RUN:
    run_experiment(name, EXPERIMENTS[name])

## Experiments table

In [ ]:
def table_row(m):
    t, v, h = m["test"], m["val"], m["history"] or {}
    return {
        "experiment": m["experiment"],
        "test_mcq_acc": t["mcq"]["acc"], "test_mcq_ci95": t["mcq"]["acc_ci95"],
        "test_mcq_single": t["mcq_by"]["single"]["acc"], "test_mcq_multi": t["mcq_by"]["multi"]["acc"],
        "test_mcq_parse_fail": t["mcq"]["parse_fail"],
        "test_saq_rougeL": t["saq"]["rougeL"], "test_saq_rougeL_ci95": t["saq"]["rougeL_ci95"],
        "test_saq_bertscore": t["saq"]["bertscore_f1"], "test_saq_bertscore_ci95": t["saq"]["bertscore_ci95"],
        "test_saq_rougeL_expert": t["saq_by"]["tier=expert"]["rougeL"],
        "test_saq_rougeL_crowd": t["saq_by"]["tier=crowdsourced"]["rougeL"],
        "test_saq_pred_words": t["saq"]["pred_words"],
        "val_mcq_acc": v["mcq"]["acc"], "val_saq_rougeL": v["saq"]["rougeL"],
        "best_epoch": h.get("best_epoch"), "best_val_loss": h.get("best_val_loss"),
        "train_min": h.get("train_minutes"),
        "bertscore_error": t.get("bertscore_error"),
    }

rows = []
for name in EXPERIMENTS:
    p = f"{OUT_ROOT}/{name}/metrics.json"
    if os.path.exists(p):
        rows.append(table_row(json.load(open(p))))
TABLE = pd.DataFrame(rows)
TABLE.to_csv(f"{OUT_ROOT}/experiments_table.csv", index=False)
pd.set_option("display.max_columns", None)
TABLE

## Good and bad answers (test): fine-tuned vs zero-shot, side by side

In [ ]:
SHOW, AGAINST, K = "E1_lora_0.5B_saq+mcq", "E0_zeroshot_0.5B", 3

def load_preds(name, split="test"):
    p = f"{OUT_ROOT}/{name}/predictions_{split}.jsonl"
    return {r["id"]: r for r in load_jsonl(p)} if os.path.exists(p) else {}

ft, zs = load_preds(SHOW), load_preds(AGAINST)
if ft:
    rng = random.Random(0)
    pick = lambda xs: rng.sample(xs, min(K, len(xs)))
    def show(r):
        print("Q:", r["question"][:400])
        if r["question_type"] == "mcq":
            print("   options:", r["options"]); print("   gold:", r["gold"], "| fine-tuned:", r["prediction"][:80],
                                                    "| zero-shot:", zs.get(r["id"], {}).get("prediction", "")[:120])
        else:
            print("   reference :", r["reference"][:300])
            print(f"   fine-tuned: {r['prediction'][:300]}   (ROUGE-L {r['rougeL']})")
            print("   zero-shot :", zs.get(r["id"], {}).get("prediction", "")[:300])
        print()
    mcq = [r for r in ft.values() if r["question_type"] == "mcq"]
    saq = sorted([r for r in ft.values() if r["question_type"] == "saq"], key=lambda r: r["rougeL"])
    print("##### MCQ correct"); [show(r) for r in pick([r for r in mcq if r["correct"]])]
    print("##### MCQ wrong");   [show(r) for r in pick([r for r in mcq if not r["correct"]])]
    print("##### SAQ highest ROUGE-L"); [show(r) for r in saq[-K:]]
    print("##### SAQ lowest ROUGE-L");  [show(r) for r in saq[:K]]
else:
    print(f"{SHOW} has no predictions yet")